# Lesson 4 - Checking for hallucinations using NLI 
# 用NLI检查幻觉

Start by setting up the notebook to minimize warnings:

从设置笔记本开始，尽量减少警告：

In [1]:
# Warning control
import warnings
warnings.filterwarnings("ignore")

Import OpenAI client and helpers to set up RAG chatbot and vector database:

导入OpenAI客户端和助手，建立RAG聊天机器人和矢量数据库：

In [2]:
import nltk
import os
nltk.download("punkt", quiet=True)

True

In [3]:
import importlib
import helper

importlib.reload(helper)
from helper import RAGChatWidget, SimpleVectorDB, get_qwen_client, load_env, download_model, download_model_with_mirror, check_model_with_mirror

Set up the client, vector database, and system message for the chatbot:

设置聊天机器人的客户端、矢量数据库和系统消息：

**prompt** ：你是阿尔弗雷多披萨咖啡馆的客服聊天机器人。您的回答应完全基于所提供的信息。

这是你的指示：

角色和行为

-你是一个友好和乐于助人的客户支持代表阿尔弗雷多的披萨咖啡馆。

-只回答与Alfredo's Pizza Cafe的菜单、网站账户管理、送货时间和其他直接相关的问题。

-不要谈论其他披萨连锁店或餐馆。

-不要回答与阿尔弗雷多披萨店或其服务无关的问题。

知识局限：

-仅使用上述知识库中提供的信息。

-如果一个问题不能使用知识库中的信息来回答，礼貌地声明你没有这些信息，并提供给用户与人工代表联系。

—请勿编造或推断知识库中未明确说明的信息。

In [4]:
prompt="""你是阿尔弗雷多披萨咖啡馆的客服聊天机器人。您的回答应完全基于所提供的信息。

这是你的指示：

角色和行为

-你是一个友好和乐于助人的客户支持代表阿尔弗雷多的披萨咖啡馆。

-只回答与Alfredo's Pizza Cafe的菜单、网站账户管理、送货时间和其他直接相关的问题。

-不要谈论其他披萨连锁店或餐馆。

-不要回答与阿尔弗雷多披萨店或其服务无关的问题。

知识局限：

-仅使用上述知识库中提供的信息。

-如果一个问题不能使用知识库中的信息来回答，礼貌地声明你没有这些信息，并提供给用户与人工代表联系。

—请勿编造或推断知识库中未明确说明的信息。"""

In [6]:
# Setup an OpenAI client
unguarded_client = get_qwen_client()

# Load up our documents that make up the knowledge base
vector_db = SimpleVectorDB.from_files("shared_data/")

# Setup system message
system_message = """You are a customer support chatbot for Alfredo's Pizza Cafe. Your responses should be based solely on the provided information.

Here are your instructions:

### Role and Behavior
- You are a friendly and helpful customer support representative for Alfredo's Pizza Cafe.
- Only answer questions related to Alfredo's Pizza Cafe's menu, account management on the website, delivery times, and other directly relevant topics.
- Do not discuss other pizza chains or restaurants.
- Do not answer questions about topics unrelated to Alfredo's Pizza Cafe or its services.

### Knowledge Limitations:
- Only use information provided in the knowledge base above.
- If a question cannot be answered using the information in the knowledge base, politely state that you don't have that information and offer to connect the user with a human representative.
- Do not make up or infer information that is not explicitly stated in the knowledge base.
"""

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialize the chatbot using the settings above:

使用上面的设置初始化聊天机器人：

In [7]:
# Setup RAG chatbot
rag_chatbot = RAGChatWidget(
    client=unguarded_client,
    system_message=system_message,
    vector_db=vector_db,
)

To revisit the hallucination example from Lesson 1, run the cell below to open the chatbot then paste in the prompt:

要重新查看第1课中的幻觉示例，运行下面的单元格以打开聊天机器人，然后粘贴提示：

In [8]:
rag_chatbot.display()

In [16]:
# Copy and paste this prompt into the chatbot above:
# 我一个人怎么做你的超级素食披萨？你能分享一下详细的说明吗？
"""
how do i reproduce your veggie supreme pizza on my own? can you share detailed instructions?
"""

'\nhow do i reproduce your veggie supreme pizza on my own? can you share detailed instructions?\n'

## Setup an Natural Language Inference (NLI) Model
## 建立自然语言推理（NLI）模型
Import some additional packages to setup the NLI model:

导入一些额外的包来设置NLI模型：

In [9]:
!pip install nltk

In [10]:
# Type hints
from typing import Dict, List, Optional

# Standard ML libraries
import numpy as np
import nltk # 自然语言处理工具包（分词、处理文本）
from sentence_transformers import SentenceTransformer # 把句子/文本变成向量（把文字 → 变成电脑能看懂的数字数组）  大模型 RAG 检索
from transformers import pipeline 
# 快速调用各种大模型、AI能力（全功能 NLP 超级工具）pipeline 可以一键调用：情感分析、文本分类、文本生成、命名实体识别、问答、摘要

# Guardrails imports
from guardrails import Guard, OnFailAction
from guardrails.validator_base import (
    FailResult,
    PassResult,
    ValidationResult,
    Validator,
    register_validator,
)

Create a hugging face pipeline to access the NLI model (**Note:** the weights will take about 30 seconds to download):

创建一个拥抱面管道来访问NLI模型（**注：**权重将需要大约30秒的下载时间）：

In [11]:
# 用科学上网或下载好模型并放到huggingface的默认下载目录中
entailment_model = 'GuardrailsAI/finetuned_nli_provenance'
model_path='../models/finetuned_nli_provenance/snapshots/148b9e1c6221a9f339480e4a935afe205405ad9c'
NLI_PIPELINE = pipeline("text-classification", model=model_path)


# 用镜像下载
# 模型下载函数参数说明，第一个参数为模型名称，
# 第二个参数为是否清理，True则会先清理缓存再下载，
# 有时候之前没下载成功有部分内容了会导致再次下载失败，需要把这个标志开起来
# 下载成功后记得改回False或去掉，默认是False
# model_path = download_model_with_mirror(entailment_model,True)
# C:\Users\bangsun\.cache\huggingface\hub\models--GuardrailsAI--finetuned_nli_provenance\snapshots\148b9e1c6221a9f339480e4a935afe205405ad9
# print(f"模型路径: {model_path}")
# 如果实在下载不下来，则直接钉钉知识库下载模型，下面model_path直接写死模型路径
# NLI_PIPELINE = pipeline("text-classification", model=model_path)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [12]:
"""finetuned_nli_provenance 模型的核心作用
这个模型是 GuardrailsAI 针对 NLI（自然语言推理）任务微调的专用模型，核心功能是判断两句话之间的逻辑关系，在你的 AI 安全护栏课程里，它是实现大模型输出真实性、合规性校验的关键工具。
1. 核心能力：NLI 自然语言推理
它能对 ** 前提（Premise）和假设（Hypothesis）** 两句话，输出三种逻辑关系：
        entailment（蕴含）：假设是前提的合理推导（两句话意思一致）
        contradiction（矛盾）：假设和前提相互冲突（意思相反）
        neutral（中立）：两句话无逻辑关联

3. 和普通 NLI 模型的区别
它是 GuardrailsAI 专门针对大模型安全校验场景微调的，相比通用 NLI 模型：
        对违规话题、虚假信息的检测更精准
        适配大模型对话场景的文本特征
        输出的分数更贴合安全护栏的判定需求

"""

'finetuned_nli_provenance 模型的核心作用\n这个模型是 GuardrailsAI 针对 NLI（自然语言推理）任务微调的专用模型，核心功能是判断两句话之间的逻辑关系，在你的 AI 安全护栏课程里，它是实现大模型输出真实性、合规性校验的关键工具。\n1. 核心能力：NLI 自然语言推理\n它能对 ** 前提（Premise）和假设（Hypothesis）** 两句话，输出三种逻辑关系：\n        entailment（蕴含）：假设是前提的合理推导（两句话意思一致）\n        contradiction（矛盾）：假设和前提相互冲突（意思相反）\n        neutral（中立）：两句话无逻辑关联\n\n3. 和普通 NLI 模型的区别\n它是 GuardrailsAI 专门针对大模型安全校验场景微调的，相比通用 NLI 模型：\n        对违规话题、虚假信息的检测更精准\n        适配大模型对话场景的文本特征\n        输出的分数更贴合安全护栏的判定需求\n\n'

Try out the pipeline:

尝试管道：

In [13]:
# Example 1: Entailed sentence
premise = "The sun rises in the east and sets in the west."
hypothesis = "The sun rises in the east."
result = NLI_PIPELINE({'text': premise, 'text_pair': hypothesis})
print(f"Example of an entailed sentence:\n\tPremise: {premise}\n\tHypothesis: {hypothesis}\n\tResult: {result}\n\n")

Example of an entailed sentence:
	Premise: The sun rises in the east and sets in the west.
	Hypothesis: The sun rises in the east.
	Result: {'label': 'entailment', 'score': 0.8697448372840881}




In [14]:
# Example 2: Contradictory sentence
premise = "The sun rises in the east and sets in the west."
hypothesis = "The sun rises in the west."
result = NLI_PIPELINE({'text': premise, 'text_pair': hypothesis})
print(f"Example of a contradictory sentence:\n\tPremise: {premise}\n\tHypothesis: {hypothesis}\n\tResult: {result}")

Example of a contradictory sentence:
	Premise: The sun rises in the east and sets in the west.
	Hypothesis: The sun rises in the west.
	Result: {'label': 'contradiction', 'score': 0.8648261427879333}


## Building a Hallucination Validator
## 构建幻觉验证器

In this section, you'll build a validator to test for hallucinations in the responses of your RAG chatbot. The validator will check that the response is grounded in the texts of your vector database.

在本节中，您将构建一个验证器来测试您的RAG聊天机器人的响应中的幻觉。验证器将检查响应是否基于矢量数据库的文本。

Start by setting up a validator with stubs for the `__init__` and `validate` functions:

首先为‘ __init__ ’和‘ validate ’函数设置一个带有存根的验证器

In [15]:
@register_validator(name="hallucination_detector", data_type="string")
class HallucinationValidation(Validator):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def validate(
        self, value: str, metadata: Optional[Dict[str, str]] = None
    ) -> ValidationResult:
        pass

Next, start fleshing out the pieces of the validator. Start by building the function that will split the response of the LLM into individual sentences:

接下来，开始充实验证器的各个部分。首先构建将LLM的响应分解为单个句子的函数：

In [16]:
# 把大模型输出的一段话，自动拆成一句一句话
@register_validator(name="hallucination_detector", data_type="string") # 注册一个校验器
class HallucinationValidation(Validator): # 创建一个幻觉检测类
    def __init__(self, **kwargs): # 继承 Guardrails 的校验器模板
        super().__init__(**kwargs)

    def validate( # 初始化  固定写法，不用管
        self, value: str, metadata: Optional[Dict[str, str]] = None    # value：大模型的回答    metadata：参考资料/上下文
    ) -> ValidationResult:
        # Split the text into sentences
        sentences = self.split_sentences(value) # 调用方法，把回答拆成句子  "我喜欢吃苹果。苹果很健康。"→["我喜欢吃苹果。", "苹果很健康。"]
        pass

    def split_sentences(self, text: str) -> List[str]: # 把一段文字切成句子
        if nltk is None: 
            raise ImportError(
                "This validator requires the `nltk` package. "
                "Install it with `pip install nltk`, and try again."
            )

        return nltk.sent_tokenize(text) # nltk（自然语言处理库）sent_tokenize → 句子拆分

# 现在只能拆句子，不能检测幻觉！
# 真正的幻觉检测需要：
# 把回答拆成句子（现在已实现）
# 用 finetuned_nli_provenance 模型判断每句话是否符合参考资料
# 不符合 → 判定为幻觉
# 拦截 / 报错

Now finalize the logic of the validate function. You'll loop through each sentence and check if it is grounded in the texts in the vector database using the `find_relevant_sources` and `check_entailment` functions. Then update the `__init__` function to set up the needed class variables.

现在完成validate函数的逻辑。您将循环遍历每个句子，并使用‘ find_relevance _sources ’和‘ check_entailment ’函数检查它是否基于矢量数据库中的文本。然后更新‘ __init__ ’函数来设置所需的类变量。

### 下面代码解释
#### 主要功能
将输入文本拆分成句子，然后对每个句子，在提供的来源中找到最相关的部分，最后使用一个自然语言推理（NLI）模型来判断这个句子是否能被找到的来源所蕴含（entailment）。如果句子无法被来源蕴含，则被视为幻觉（hallucination），验证失败。
#### 分块解释
##### 初始化函数
- **功能说明**:：参数初始化；嵌入模型加载；NLI 模型加载；基类初始化
- **输入**:
    1. embedding_model: 用于将文本转换为向量的模型，默认使用 'all-MiniLM-L6-v2'（轻量级句子嵌入模型）。
    2. entailment_model: 用于判断“前提→假设”关系的 NLI 模型，默认使用 GuardrailsAI/finetuned_nli_provenance。
    3. sources: 提供的事实依据列表（比如知识库、上下文、文档等），用来判断生成内容是否“有据可依”。

##### validate方法
- **功能说明**：它接收一段由大型语言模型（LLM）生成的文本，并根据预先提供的参考来源（sources），逐句核查该文本的事实准确性和可追溯性。将文本拆解 → 检索相关证据 → 使用自然语言推理（NLI）模型判断每个句子是否能被证据支持。
- **输入**: 待验证的文本 `value` (一个字符串，这里是LLM返回的内容)。
- **步骤**:
    1. **拆分句子**: `sentences = self.split_sentences(value)` 将输入的文本拆分成单独的句子列表。
    2. **查找相关来源**: `relevant_sources = self.find_relevant_sources(sentences, self.sources)` 根据句子和所有来源的**语义相似度**，找到与这些句子最相关的**一小部分**来源文本。
    3. **循环验证**: 遍历每个拆分后的 `sentence`：
        - `is_entailed = self.check_entailment(sentence, relevant_sources)`：检查该句子是否能被相关来源所蕴含。
        - 根据结果将句子分别加入 `hallucinated_sentences`（幻觉）或 `entailed_sentences`（被蕴含）。
- **结果返回**:
    - 如果 `hallucinated_sentences` 列表**非空**，返回 `FailResult`（失败），并附带错误信息说明哪些句子是幻觉。
    - 如果所有句子都被蕴含，返回 `PassResult`（成功）。

##### find_relevant_sources 方法
- **功能说明**： 将待验证的文本（已拆分为句子）与提供的全部来源（sources）进行比较，通过计算它们之间的余弦相似度，来找出与每个句子语义上最相关的少数几个来源片段
- **输入**:
    1. sentences：要验证的文本切成一个个句子后的列表 (这里是LLM返回的内容切分后的雷彪)。
    2. sources：原始材料（这里相当于掉大模型前在向量库里检索出来的要传给大模型的相关片段）
- **步骤**:
    1. **向量化所有文本**:
    2. 循环每个输入的句子检索处于其最相关的来源
        - 获取当前句子的嵌入向量并转出二维数组
        - 当前句子和所有来源句子计算余弦相似度
        - 找到相似度最高的5个来源的索引
        - 进一步筛选：只保留相似度大于0.8的来源
        - 将符合条件的来源文本添加到返回列表relevant_sources中
- **结果返回**:
    + 返回: relevant_sources (一个字符串列表)。
    + 包含从原始 sources 中筛选出的、与任一待验证句子语义高度相关的文本片段。这些片段将作为 NLI 验证模型的有效证据。

In [20]:
# 注册一个自定义校验器，名字叫hallucination_detector，只校验字符串类型
@register_validator(name="hallucination_detector", data_type="string")
# 定义幻觉检测类，继承Guardrails的基础校验器Validator
class HallucinationValidation(Validator):
    # 构造函数：初始化模型、资源等配置
    def __init__(
            self, 
            embedding_model: Optional[str] = None,  # 向量模型名称，可选
            entailment_model: Optional[str] = None,  # NLI推理模型名称，可选
            sources: Optional[List[str]] = None,  # 可信参考资料，可选
            **kwargs  # 其他参数
        ):
        # 如果没有传入向量模型，使用默认的轻量级模型all-MiniLM-L6-v2
        if embedding_model is None:
            embedding_model = 'all-MiniLM-L6-v2'
        # 加载向量模型，用于计算文本相似度
        self.embedding_model = SentenceTransformer(embedding_model)

        # 保存传入的可信参考资料
        self.sources = sources
        
        # 如果没有传入NLI模型，使用Guardrails官方的推理模型
        if entailment_model is None:
            entailment_model = 'GuardrailsAI/finetuned_nli_provenance'
        # 加载NLI管道，用于判断句子是否蕴含（是否真实可信）
        model_path='../models/finetuned_nli_provenance/snapshots/148b9e1c6221a9f339480e4a935afe205405ad9c'
        self.nli_pipeline = pipeline("text-classification", model=model_path)
        # self.nli_pipeline = pipeline("text-classification", model=entailment_model)

        # 调用父类Validator的初始化方法（必须写）
        super().__init__(**kwargs)

    # 核心校验方法：value是模型生成的回答，metadata是额外上下文
    def validate(
        self, value: str, metadata: Optional[Dict[str, str]] = None
    ) -> ValidationResult:
        # 第一步：把模型回答按句子拆分
        sentences = self.split_sentences(value)

        # 第二步：为每句话找到最相关的可信参考资料
        relevant_sources = self.find_relevant_sources(sentences, self.sources)

        # 分别保存可信句子和幻觉句子
        entailed_sentences = []
        hallucinated_sentences = []
        # 遍历每一句话，检查是否是幻觉
        for sentence in sentences:
            # 第三步：检查这句话是否被参考资料蕴含（即是否真实）
            is_entailed = self.check_entailment(sentence, relevant_sources)
            # 不被蕴含 → 幻觉
            if not is_entailed:
                hallucinated_sentences.append(sentence)
            # 被蕴含 → 可信
            else:
                entailed_sentences.append(sentence)
        
        # 如果有幻觉句子 → 校验失败，返回错误
        if len(hallucinated_sentences) > 0:
            return FailResult(
                error_message=f"The following sentences are hallucinated: {hallucinated_sentences}",
            )
        
        # 没有幻觉 → 校验通过
        return PassResult()

    # 工具方法：把一段文本拆分成句子列表
    def split_sentences(self, text: str) -> List[str]:
        # 如果没安装nltk，直接报错提示安装
        if nltk is None:
            raise ImportError(
                "This validator requires the `nltk` package. "
                "Install it with `pip install nltk`, and try again."
            )
        # 使用nltk的句子拆分工具
        return nltk.sent_tokenize(text)

    # 工具方法：找到与句子最相关的参考资料（相似度>0.8）
    def find_relevant_sources(self, sentences: str, sources: List[str]) -> List[str]:
        # 把所有参考资料转成向量
        source_embeds = self.embedding_model.encode(sources)
        # 把所有句子转成向量
        sentence_embeds = self.embedding_model.encode(sentences)

        # 保存最相关的资料
        relevant_sources = []

        # 遍历每一句话
        for sentence_idx in range(len(sentences)):
            # 拿到当前句子的向量
            sentence_embed = sentence_embeds[sentence_idx, :].reshape(1, -1)
            # 计算句子与所有资料的余弦相似度
            cos_similarities = np.sum(np.multiply(source_embeds, sentence_embed), axis=1)
            # 找出相似度最高的前5个资料
            top_sources = np.argsort(cos_similarities)[::-1][:5]
            # 只保留相似度>0.8的资料
            top_sources = [i for i in top_sources if cos_similarities[i] > 0.8]

            # 把相关资料加入结果
            relevant_sources.extend([sources[i] for i in top_sources])

        return relevant_sources
    
    # 工具方法：检查句子是否被任意一个参考资料蕴含
    def check_entailment(self, sentence: str, sources: List[str]) -> bool:
        # 遍历所有参考资料
        for source in sources:
            # 用NLI模型判断：参考资料 是否蕴含 模型生成的句子
            output = self.nli_pipeline({'text': source, 'text_pair': sentence})
            # 如果标签是entailment → 可信
            if output['label'] == 'entailment':
                return True
        # 没有任何资料能支撑这句话 → 幻觉
        return False

Try out the validator. First you'll create an instance of the `HallucinationValidation` class above, passing in the same sentence as you used in the pipeline test above:

尝试验证器。首先，你将创建上面的“HallucinationValidation”类的一个实例，传入与在上面的管道测试中使用的相同的句子：

In [21]:
hallucination_validator = HallucinationValidation(
    sources = ["The sun rises in the east and sets in the west"]
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Then use the `validate()` function of this object, passing in the sentence you want to test. The first example does not entail, but the second does:

然后使用该对象的‘ validate() ’函数，传入要测试的句子。第一个例子没有限定，但第二个例子有：

In [22]:
result = hallucination_validator.validate("The sun sets in the east")
print(f"Validation outcome: {result.outcome}")
if result.outcome == "fail":
    print(f"Error message: {result.error_message}")

Validation outcome: fail
Error message: The following sentences are hallucinated: ['The sun sets in the east']


In [34]:
result = hallucination_validator.validate("The sun sets in the west")
print(f"Validation outcome: {result.outcome}")
if result.outcome == "fail":
    print(f"Error message: {result.error_message}")

Validation outcome: pass


In the next lesson, you'll build a guard around this validator. 